In [1]:
import numpy as np
import pandas as pd

In [2]:
visits = pd.read_csv('metrika_visits_test.csv')

In [3]:
visits.columns

Index(['project_id', 'visit_id', 'date_time', 'is_new_user', 'client_id',
       'region_country', 'region_city', 'watch_ids', 'traffic_source',
       'adv_engine', 'search_engine_root', 'search_engine', 'social_network',
       'recommendation_system', 'messenger', 'device_category', 'mobile_phone',
       'mobile_phone_model', 'operating_system_root', 'operating_system',
       'browser', 'browser_major_version', 'screen_width', 'screen_height'],
      dtype='object')

In [4]:
import ast
if isinstance(visits['watch_ids'].iloc[0], str):
    visits['watch_ids'] = visits['watch_ids'].apply(ast.literal_eval)

# 2. Explode the list column
# This creates a row for every watch_id in the list
visits_exploded = visits.explode('watch_ids')

# 3. Rename the column for clarity and remove any nulls
visits_exploded = visits_exploded.rename(columns={'watch_ids': 'watch_id'})
visits_exploded = visits_exploded.dropna(subset=['watch_id'])

# 4. Verify the transformation
print(f"Original unique visits: {len(visits)}")
print(f"Total rows after exploding: {len(visits_exploded)}")

Original unique visits: 1363
Total rows after exploding: 12521


In [6]:
len(visits_exploded['visit_id'].unique())

1363

In [7]:
hits = pd.read_csv('metrika_hits_test.csv')

In [8]:
len(hits['watch_id'].unique())

6880

In [9]:
# hits_prod = hits[hits['page_type'] == 'PRODUCT']
# hits_prod

In [10]:
hits = hits.drop(columns=['region_country',
                                           'region_city', 'url','artificial', 'link', 'download', 'last_traffic_source',
                                           'last_search_engine_root', 'last_search_engine', 'last_adv_engine',
                                           'last_social_network', 'last_social_network_profile',
                                           'recommendation_system', 'messenger', 'device_category', 'mobile_phone',
                                           'mobile_phone_model', 'operating_system_root', 'operating_system',
                                           'browser', 'browser_major_version', 'screen_width', 'screen_height'], axis=1)

In [11]:
old_site_p = pd.read_csv('old_site_products.csv')
new_site_p = pd.read_csv('new_site_products.csv')
cols_to_keep = ['id', 'name', 'brand', 'main_category', 'categories', 'slug']
old_site_p = old_site_p[cols_to_keep]
new_site_p = new_site_p[cols_to_keep]

In [12]:
mapping = pd.read_csv('old_site_new_site_products.csv')

In [13]:
slug_to_pid = dict(zip(new_site_p['slug'], new_site_p['id']))
cat_to_pid = dict(zip(new_site_p['main_category'], new_site_p['id']))
brand_to_pid = dict(zip(new_site_p['brand'], new_site_p['id']))

In [15]:
hits['product_id'] = hits['slug'].map(slug_to_pid)
hits.isna().sum()

project_id         0
watch_id           0
date_time          0
client_id          0
page_type          0
slug            1134
is_page_view       0
not_bounce         0
product_id      5133
dtype: int64

In [16]:
old_to_new_map = dict(zip(mapping['old_site_id'], mapping['new_site_id']))

# If the hit was on the old site, translate the ID to the new site equivalent
def translate_to_new(row):
    if row['project_id'] == 1: # Old site
        return old_to_new_map.get(row['product_id'])
    return row['product_id'] # Already a new site ID or NaN

hits['new_site_id'] = hits.apply(translate_to_new, axis=1)
hits.isna().sum()

project_id         0
watch_id           0
date_time          0
client_id          0
page_type          0
slug            1134
is_page_view       0
not_bounce         0
product_id      5133
new_site_id     5133
dtype: int64

In [15]:
hits_prod = hits_prod.dropna()
hits_prod.isna().sum()

project_id      0
watch_id        0
date_time       0
client_id       0
page_type       0
slug            0
is_page_view    0
not_bounce      0
product_id      0
new_site_id     0
dtype: int64

In [39]:
t_hits = pd.read_csv('metrika_hits_test.csv')
t_visits = pd.read_csv('metrika_visits_test.csv')

# Convert to datetime
t_hits['date_time'] = pd.to_datetime(t_hits['date_time'], format='ISO8601')
t_visits['date_time'] = pd.to_datetime(t_visits['date_time'], format='ISO8601')

# This merge catches hits that occur slightly before the official 'visit' start
merged_test = pd.merge_asof(
    t_hits.sort_values('date_time'),
    t_visits.sort_values('date_time').rename(columns={'date_time': 'visit_start'}),
    by='client_id',
    left_on='date_time',
    right_on='visit_start'
)

In [40]:
merged_test['product_id'] = merged_test['slug'].map(slug_to_pid)
merged_test.isna().sum()

project_id_x                      0
watch_id                          0
date_time                         0
client_id                         0
region_country_x                 65
region_city_x                  1066
url                               0
page_type                         0
slug                           1134
artificial                        0
is_page_view                      0
not_bounce                        0
link                              0
download                          0
last_traffic_source               0
last_search_engine_root        5888
last_search_engine             5888
last_adv_engine                6233
last_social_network            6872
last_social_network_profile    6880
recommendation_system_x        6856
messenger_x                    6880
device_category_x                 0
mobile_phone_x                 1226
mobile_phone_model_x           1217
operating_system_root_x           0
operating_system_x                0
browser_x                   

In [41]:
merged_test['slug'].isin(enriched_hits['slug']).sum()

5915

In [42]:
enriched_hits = hits.merge(
    visits_exploded[['date_time', 'watch_id', 'client_id', 'visit_id']], 
    on='date_time', 
    how='inner'
)
enriched_hits.shape

(12117, 13)

In [43]:
# merged_test = merged_test.drop(columns=['region_country',
#                                            'region_city', 'url','artificial', 'link', 'download', 'last_traffic_source',
#                                            'last_search_engine_root', 'last_search_engine', 'last_adv_engine',
#                                            'last_social_network', 'last_social_network_profile',
#                                            'recommendation_system', 'messenger', 'device_category', 'mobile_phone',
#                                            'mobile_phone_model', 'operating_system_root', 'operating_system',
#                                            'browser', 'browser_major_version', 'screen_width', 'screen_height'], axis=1)

In [44]:
merged_test = merged_test.sort_values(['visit_id', 'date_time'])

# 2. Group into lists of sequences
sequences = merged_test.groupby('visit_id').agg({'product_id': list})
sequences

,product_id
visit_id,
0000913076882209575,"[463480358.0, 463480358.0, nan]"
0002268743573412674,"[nan, 463480242.0, 463480242.0, nan, 463480242..."
0017209440033371282,"[nan, nan, nan]"
0028175944123250496,"[nan, nan]"
0031343418629767038,"[nan, nan, nan, nan, nan, 495256989.0, 4952590..."
...,...
9947592951664175637,"[nan, nan, nan, nan]"
9968262306115660178,"[nan, nan]"
9968912483060354231,"[nan, 463480242.0]"


In [47]:
sequences.to_csv('co_occurence_test_sequence.csv')

In [12]:
# temp_df = enriched_hits[~enriched_hits['slug'].isna()]
# temp_df

In [13]:
# unified_slug_to_id = dict(zip(unified_catalog['slug'], unified_catalog['id']))
# len(unified_slug_to_id)

In [14]:
# temp_df['slug'].apply(lambda x: x not in unified_slug_to_id.keys()).sum()

In [15]:
old_site_p = pd.read_csv('old_site_products.csv')
new_site_p = pd.read_csv('new_site_products.csv')

In [16]:
cols_to_keep = ['id', 'name', 'brand', 'main_category', 'categories', 'slug']
old_site_p = old_site_p[cols_to_keep]
new_site_p = new_site_p[cols_to_keep]

In [17]:
old_new_concat = pd.concat([old_site_p, new_site_p])
old_new_concat

,id,name,brand,main_category,categories,slug
0,3259,Автокресло Baby Care BSO Sport Isofix,Baby Care,Автокресла,"[""Автокресла"",""Прогулки и путешествия""]",avtokreslo-baby-care-bso-sport-isofix
1,3261,Автокресло Chicco Key1 X-Plus ISOFIX,Chicсo,Автокресла,"[""Автокресла"",""Прогулки и путешествия""]",avtokreslo-chicco-key1-x-plus-isofix
2,3262,Автокресло Chicco Synthesis XT-Plus,Chicсo,Автокресла,"[""Автокресла"",""Прогулки и путешествия""]",avtokreslo-chicco-synthesis-xt-plus
3,3265,Дом-палатка с туннелем «Океан»,Ching-Ching,Домики и палатки,"[""Домики и палатки"",""Тренажеры и игровые компл...",dom-palatka-s-tunnelem-okean
4,3267,Барабан для малышей,Imaginarium,NaN,[],baraban-dlya-malyshey
...,...,...,...,...,...,...
683,1651502137,Адаптеры для автолюльки Yoyo,BabyZen,Коляски,"[""Коляски""]",adaptery-dlya-avtolyulki-yoyo
684,1655011689,Электрокачели MamaRoo NEW + весы Maman напрокат,Maman##4moms,Акции недели,"[""Акции недели"",""Качели, шезлонги"",""Весы""]",elektrokacheli-mamaroo-new-vesy-maman-naprokat
685,1677458905,Обезьянка на кольцах напрокат,Bright Starts,Музыкальные игрушки,"[""Развивающие игрушки"",""Музыкальные игрушки""]",obezyanka-na-koltsah-naprokat
686,1677460825,"Развивающая игрушка ""Маленькие друзья"" напрокат",Fisher-Price,Музыкальные игрушки,"[""Развивающие игрушки"",""Музыкальные игрушки""]",razvivayuschaya-igrushka-malenkie-druzya-naprokat


In [18]:
enriched_hits_merged = enriched_hits.merge(old_new_concat, on='slug', how='left')
print(enriched_hits_merged.shape)
print(enriched_hits_merged.isna().sum())

(7210, 15)
project_id          0
watch_id            0
date_time           0
client_id           0
page_type           0
slug             1134
is_page_view        0
not_bounce          0
visit_id            0
is_new_user         0
id               5119
name             5119
brand            5156
main_category    5122
categories       5119
dtype: int64


In [19]:
# temp_enr_hit = enriched_hits[~enriched_hits['slug'].isna()]
# temp_enr_hit.isna().sum()

In [20]:
# temp_enr_hit['date_time'] = pd.to_datetime(temp_enr_hit['date_time'], format='ISO8601')

# temp_enr_hit = temp_enr_hit.sort_values(['visit_id', 'date_time'])

# # 2. Group by visit_id to see the sequence of URLs/Slugs
# user_journeys_new = temp_enr_hit.groupby(['client_id', 'visit_id'])['slug'].apply(list).reset_index()

# # 3. View a sample journey
# print(user_journeys_new.head())

In [21]:
enriched_hits_merged

,project_id,watch_id,date_time,client_id,page_type,slug,is_page_view,not_bounce,visit_id,is_new_user,id,name,brand,main_category,categories
0,0,6034151852304141635,2025-10-15T02:13:22,911978007540833652,CATEGORY,kacheli-shezlongi,1,0,6034151852304141635,1,NaN,NaN,NaN,NaN,NaN
1,0,3732462510408219172,2025-10-15T02:13:37,911978007540833652,CATEGORY,kacheli-shezlongi,0,1,6034151852304141635,1,NaN,NaN,NaN,NaN,NaN
2,0,0473312698303184850,2025-10-15T08:35:23,4254606298939301523,CATEGORY,kokony-dlya-novorozhdennyh,1,0,0473312698303184850,1,NaN,NaN,NaN,NaN,NaN
3,0,1378704074885058207,2025-10-15T08:35:38,4254606298939301523,CATEGORY,kokony-dlya-novorozhdennyh,0,1,0473312698303184850,1,NaN,NaN,NaN,NaN,NaN
4,0,0561761801888023325,2025-10-15T08:35:44,4254606298939301523,CATEGORY,razvivayuschie-igrushki,1,0,0473312698303184850,1,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7205,0,10477861506005733632,2026-01-05T19:34:32,5942727998630520181,CATEGORY,avtokresla-avtolyulki,1,0,25410166595495903928,1,NaN,NaN,NaN,NaN,NaN
7206,0,05985114779355722526,2026-01-05T19:35:09,5942727998630520181,CATEGORY,avtokresla-avtolyulki,1,0,25410166595495903928,1,NaN,NaN,NaN,NaN,NaN
7207,0,50474572206340698578,2026-01-05T23:55:49,6469027861517018798,MAIN,NaN,1,0,50474572206340698578,0,NaN,NaN,NaN,NaN,NaN
7208,0,79082114192378376478,2026-01-05T23:56:04,6469027861517018798,MAIN,NaN,0,1,50474572206340698578,0,NaN,NaN,NaN,NaN,NaN


In [22]:
enriched_hits_merged['slug'] = enriched_hits_merged['slug'].fillna('UNKNOWN')
enriched_hits_merged[enriched_hits_merged['slug'] == 'UNKNOWN']

,project_id,watch_id,date_time,client_id,page_type,slug,is_page_view,not_bounce,visit_id,is_new_user,id,name,brand,main_category,categories
21,0,3031295072759772554,2025-10-15T08:41:17,4254606298939301523,CART,UNKNOWN,1,0,0473312698303184850,1,NaN,NaN,NaN,NaN,NaN
52,0,3836721908992353636,2025-10-15T10:37:06,9299171188988743287,MAIN,UNKNOWN,1,0,1835293676913553579,0,NaN,NaN,NaN,NaN,NaN
55,0,2176772772464915447,2025-10-15T11:40:43,4648167420337642555,CART,UNKNOWN,1,0,9688455479360508107,1,NaN,NaN,NaN,NaN,NaN
56,0,5864744113543991733,2025-10-15T11:40:51,4648167420337642555,CHECKOUT,UNKNOWN,1,0,9688455479360508107,1,NaN,NaN,NaN,NaN,NaN
57,0,0128238365881211176,2025-10-15T11:42:57,4648167420337642555,CHECKOUT,UNKNOWN,0,0,9688455479360508107,1,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7193,0,23980720595766667257,2026-01-05T13:06:47,9289751290677769668,MAIN,UNKNOWN,1,0,13205693492865993598,0,NaN,NaN,NaN,NaN,NaN
7197,0,13878944487482562098,2026-01-05T16:36:30,2010106643374012304,MAIN,UNKNOWN,1,0,13878944487482562098,1,NaN,NaN,NaN,NaN,NaN
7198,0,98680595890697072479,2026-01-05T16:36:42,2010106643374012304,CART,UNKNOWN,1,0,13878944487482562098,1,NaN,NaN,NaN,NaN,NaN
7207,0,50474572206340698578,2026-01-05T23:55:49,6469027861517018798,MAIN,UNKNOWN,1,0,50474572206340698578,0,NaN,NaN,NaN,NaN,NaN


In [56]:
enriched_hits_merged['date_time'] = pd.to_datetime(enriched_hits_merged['date_time'], format='ISO8601')

enriched_hits_merged = enriched_hits_merged.sort_values(['visit_id', 'date_time'])

# 2. Group by visit_id to see the sequence of URLs/Slugs
user_journeys = enriched_hits_merged.groupby(['visit_id']).agg({'slug' : list, 'id': list}).reset_index()

# 3. View a sample journey
print(user_journeys.shape)

(1363, 3)


In [57]:
user_journeys['slug'].apply(lambda x: len(x) != 1).sum()

1363

In [58]:
user_journeys.drop(columns = ['id'], axis=1, inplace=True)
user_journeys

,visit_id,slug
0,0000913076882209575,"[manezh-hauck-dreamn-play-naprokat, manezh-hau..."
1,0002268743573412674,"[avtokresla-avtolyulki, avtokreslo-britax-roem..."
2,0017209440033371282,"[UNKNOWN, vesy, vesy-sasha-kokon-farla-baby-sh..."
3,0028175944123250496,"[montessori, montessori]"
4,0031343418629767038,"[UNKNOWN, UNKNOWN, avtokresla-avtolyulki, avto..."
...,...,...
1358,9947592951664175637,"[kolyaski, kolyaski, kolyaski, kolyaski]"
1359,9968262306115660178,"[kacheli-shezlongi, kacheli-shezlongi]"
1360,9968912483060354231,"[avtokresla-avtolyulki, avtokreslo-britax-roem..."
1361,9992384201760487821,"[avtokresla-avtolyulki, avtokresla-avtolyulki,..."


In [59]:
user_journeys.to_csv('user_journeys_test.csv', index=False)

In [95]:
enriched_hits_merged

,project_id,watch_id,date_time,client_id,page_type,slug,is_page_view,not_bounce,visit_id,is_new_user,id,name,brand,main_category,categories
4621,0,0000913076882209575,2025-12-09 12:23:12,51760759414751783,PRODUCT,manezh-hauck-dreamn-play-naprokat,1,0,0000913076882209575,0,463480358.0,Манеж Hauck Dream’n Play напрокат,Hauck,"Кроватки, манежи","[""Кроватки, манежи""]"
4622,0,7718010160368323835,2025-12-09 12:23:27,51760759414751783,PRODUCT,manezh-hauck-dreamn-play-naprokat,0,1,0000913076882209575,0,463480358.0,Манеж Hauck Dream’n Play напрокат,Hauck,"Кроватки, манежи","[""Кроватки, манежи""]"
4623,0,1360665076774416389,2025-12-09 12:24:00,51760759414751783,CATEGORY,krovatki-manezhi,1,0,0000913076882209575,0,NaN,NaN,NaN,NaN,NaN
6273,0,0002268743573412674,2025-12-25 12:12:44,225462741523031013,CATEGORY,avtokresla-avtolyulki,1,0,0002268743573412674,1,NaN,NaN,NaN,NaN,NaN
6274,0,0830813214661936766,2025-12-25 12:13:00,225462741523031013,PRODUCT,avtokreslo-britax-roemer-kidfix-sl-naprokat,1,0,0002268743573412674,1,463480242.0,Автокресло Britax Roemer Kidfix SL (15-36 кг) ...,Britax Römer,Автокресла Britax Romer,"[""Автокресла, автолюльки"",""Автокресла Britax R..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2362,0,7777011767880430609,2025-11-14 13:32:09,9062815272629922906,CHECKOUT,UNKNOWN,0,0,9992384201760487821,1,NaN,NaN,NaN,NaN,NaN
2363,0,1185458203648096349,2025-11-14 13:32:09,9062815272629922906,CHECKOUT,UNKNOWN,0,0,9992384201760487821,1,NaN,NaN,NaN,NaN,NaN
2364,0,6798132504543439958,2025-11-14 13:32:11,9062815272629922906,ORDER,eef3bcbb381a62f2e0221196ae3e6079,1,0,9992384201760487821,1,NaN,NaN,NaN,NaN,NaN
150,0,9994141008995322002,2025-10-16 17:22:35,7138618447530126546,CATEGORY,kolyaski,1,0,9994141008995322002,1,NaN,NaN,NaN,NaN,NaN


# Unique Slugs Dataframe

In [49]:
unique_slugs_df = pd.DataFrame(enriched_hits_merged['slug'].unique(), columns=['slug'])


unique_slugs_df['pid_is_available'] = unique_slugs_df['slug'].isin(merged_dict.keys()).astype(int)

print(unique_slugs_df.head())

                                                slug  pid_is_available
0                  manezh-hauck-dreamn-play-naprokat                 1
1                                   krovatki-manezhi                 0
2                              avtokresla-avtolyulki                 0
3        avtokreslo-britax-roemer-kidfix-sl-naprokat                 1
4  avtokreslo-gruppa-23-15-36-kg-recaro-milano-se...                 1


In [50]:
unique_slugs_df['pid'] = unique_slugs_df['slug'].map(merged_dict)
unique_slugs_df

,slug,pid_is_available,pid
0,manezh-hauck-dreamn-play-naprokat,1,463480358.0
1,krovatki-manezhi,0,NaN
2,avtokresla-avtolyulki,0,NaN
3,avtokreslo-britax-roemer-kidfix-sl-naprokat,1,463480242.0
4,avtokreslo-gruppa-23-15-36-kg-recaro-milano-se...,1,463480342.0
...,...,...,...
360,kacheli-shezlong-4moms-mamaroo-40-grafitovyy-m...,1,463480491.0
361,8649fa08e6c9c26fde3d872c72ee7779,0,NaN
362,katalka-tolokar-fisher-price-lvenok-naprokat,1,463480441.0
363,a440b6543a131cd4cd04cfac07ff0787,0,NaN


In [51]:
unique_slugs_df['pid'] = unique_slugs_df['pid'].fillna(-1)
unique_slugs_df

,slug,pid_is_available,pid
0,manezh-hauck-dreamn-play-naprokat,1,463480358.0
1,krovatki-manezhi,0,-1.0
2,avtokresla-avtolyulki,0,-1.0
3,avtokreslo-britax-roemer-kidfix-sl-naprokat,1,463480242.0
4,avtokreslo-gruppa-23-15-36-kg-recaro-milano-se...,1,463480342.0
...,...,...,...
360,kacheli-shezlong-4moms-mamaroo-40-grafitovyy-m...,1,463480491.0
361,8649fa08e6c9c26fde3d872c72ee7779,0,-1.0
362,katalka-tolokar-fisher-price-lvenok-naprokat,1,463480441.0
363,a440b6543a131cd4cd04cfac07ff0787,0,-1.0


In [52]:
match_count = unique_slugs_df['pid_is_available'].sum()
print(f"Matched: {match_count} / {len(unique_slugs_df)}")

Matched: 254 / 365


In [53]:
unique_slugs_df = unique_slugs_df[~unique_slugs_df['slug'].isna()]
unique_slugs_df

,slug,pid_is_available,pid
0,manezh-hauck-dreamn-play-naprokat,1,463480358.0
1,krovatki-manezhi,0,-1.0
2,avtokresla-avtolyulki,0,-1.0
3,avtokreslo-britax-roemer-kidfix-sl-naprokat,1,463480242.0
4,avtokreslo-gruppa-23-15-36-kg-recaro-milano-se...,1,463480342.0
...,...,...,...
360,kacheli-shezlong-4moms-mamaroo-40-grafitovyy-m...,1,463480491.0
361,8649fa08e6c9c26fde3d872c72ee7779,0,-1.0
362,katalka-tolokar-fisher-price-lvenok-naprokat,1,463480441.0
363,a440b6543a131cd4cd04cfac07ff0787,0,-1.0


In [54]:
unique_slugs_df.to_csv('unique_slugs_test_df.csv', index = False)

In [55]:
unique_slugs_df['slug'].unique().shape

(365,)

In [38]:
unique_slugs_embeddings = pd.read_csv('unique_slugs_embeddings.csv')

In [39]:
unique_slugs_embeddings

,Unnamed: 0,slug,pid_is_available,embeddings,product_id
0,0,ROOT,0,[ 1.56962778e-02 -4.57994342e-02 1.84462667e-...,-1.0
1,1,manezhi-i-krovatki,0,[ 2.66767032e-02 -1.90703776e-02 3.84690240e-...,-1.0
2,2,krovatka-chicco-next2me,1,[ 0.05553827 -0.08089578 0.03729409 -0.016796...,3572.0
3,3,krovatka-manezh-chicco-zipgo,1,[ 2.78634150e-02 -8.13705251e-02 6.06315248e-...,3466.0
4,4,skladnaya-krovatka-snuggle-nest-surround-khl,1,[ 1.00846952e-02 -7.04894662e-02 4.13043872e-...,3825.0
...,...,...,...,...,...
723,723,gamak-krovatka-kidwood,1,[ 4.47303765e-02 -8.11619237e-02 1.22936862e-...,3387.0
724,724,avtokreslo-maxi-cosi-pebble-crazy-mamas,1,[ 3.21396068e-02 -6.85035065e-02 7.52658993e-...,6667.0
725,725,gonki-elc,1,[ 3.71554382e-02 -6.34240359e-02 -1.09364707e-...,3409.0
726,726,mobil-druzya-iz-tropicheskogo-lesa-popugay,1,[ 3.88050713e-02 -1.89655311e-02 5.25056347e-...,495519046.0


In [40]:
unique_slugs_embeddings['product_id'] = unique_slugs_embeddings['slug'].map(merged_dict)
unique_slugs_embeddings

,Unnamed: 0,slug,pid_is_available,embeddings,product_id
0,0,ROOT,0,[ 1.56962778e-02 -4.57994342e-02 1.84462667e-...,NaN
1,1,manezhi-i-krovatki,0,[ 2.66767032e-02 -1.90703776e-02 3.84690240e-...,NaN
2,2,krovatka-chicco-next2me,1,[ 0.05553827 -0.08089578 0.03729409 -0.016796...,3572.0
3,3,krovatka-manezh-chicco-zipgo,1,[ 2.78634150e-02 -8.13705251e-02 6.06315248e-...,3466.0
4,4,skladnaya-krovatka-snuggle-nest-surround-khl,1,[ 1.00846952e-02 -7.04894662e-02 4.13043872e-...,3825.0
...,...,...,...,...,...
723,723,gamak-krovatka-kidwood,1,[ 4.47303765e-02 -8.11619237e-02 1.22936862e-...,3387.0
724,724,avtokreslo-maxi-cosi-pebble-crazy-mamas,1,[ 3.21396068e-02 -6.85035065e-02 7.52658993e-...,6667.0
725,725,gonki-elc,1,[ 3.71554382e-02 -6.34240359e-02 -1.09364707e-...,3409.0
726,726,mobil-druzya-iz-tropicheskogo-lesa-popugay,1,[ 3.88050713e-02 -1.89655311e-02 5.25056347e-...,495519046.0


In [ ]:
unique_slugs_embeddings['product_id'] = unique_slugs_embeddings['product_id'].fillna(-1)
unique_slugs_embeddings

In [ ]:
unique_slugs_embeddings.to_csv('unique_slugs_embeddings.csv', index=False)

In [60]:
user_journeys

,visit_id,slug
0,0000913076882209575,"[manezh-hauck-dreamn-play-naprokat, manezh-hau..."
1,0002268743573412674,"[avtokresla-avtolyulki, avtokreslo-britax-roem..."
2,0017209440033371282,"[UNKNOWN, vesy, vesy-sasha-kokon-farla-baby-sh..."
3,0028175944123250496,"[montessori, montessori]"
4,0031343418629767038,"[UNKNOWN, UNKNOWN, avtokresla-avtolyulki, avto..."
...,...,...
1358,9947592951664175637,"[kolyaski, kolyaski, kolyaski, kolyaski]"
1359,9968262306115660178,"[kacheli-shezlongi, kacheli-shezlongi]"
1360,9968912483060354231,"[avtokresla-avtolyulki, avtokreslo-britax-roem..."
1361,9992384201760487821,"[avtokresla-avtolyulki, avtokresla-avtolyulki,..."


In [62]:
old_sl = dict(zip(old_site_p['slug'], old_site_p['id']))
new_sl = dict(zip(new_site_p['slug'], new_site_p['id']))

In [63]:
new_sl

{'matras-red-castle-kokon-dlya-novorozhdennyh-cocoonababy-naprokat': 463480210,
 'kokon-dlya-novorozhdennyh-matello-cocon-baby-lux-naprokat': 463480211,
 'kokon-lyulka-dlya-novorozhdennyh-farla-baby-shell-naprokat': 463480212,
 'kacheli-shezlong-4moms-mamaroo-40-naprokat': 463480213,
 'kacheli-shezlong-4moms-mamaroo-30-naprokat': 463480214,
 'detskie-eletrokacheli-pituso-osito-naprokat': 463480215,
 'kacheli-bright-starts-cradling-swing': 463480216,
 'molokootsos-medela-swing-maxi-double-naprokat': 463480217,
 'molokootsos-medela-freestyle-double-naprokat': 463480218,
 'molokootsos-medela-swing-single-naprokat': 463480219,
 'avtokreslo-happybaby-gelios-v2-naprokat': 463480220,
 'avtokreslo-britax-romer-baby-safe-naprokat': 463480221,
 'avtokreslo-cybex-aton-q-plus-naprokat': 463480222,
 'vesy-dlya-novorozhdennyh-maman-vend-naprokat': 463480223,
 'vesy-dlya-novorozhdyonnyh-laica-ps3003-naprokat': 463480224,
 'vesy-dlya-novorozhdennyh-laica-ps3001-naprokat': 463480225,
 'detskaya-kolyask

In [71]:
en = set(enriched_hits['slug'].unique())
new = set(new_sl)
matched = en.intersection(new)
print(len(matched))

253


In [72]:
old_site_p

,id,name,brand,main_category,categories,slug
0,3259,Автокресло Baby Care BSO Sport Isofix,Baby Care,Автокресла,"[""Автокресла"",""Прогулки и путешествия""]",avtokreslo-baby-care-bso-sport-isofix
1,3261,Автокресло Chicco Key1 X-Plus ISOFIX,Chicсo,Автокресла,"[""Автокресла"",""Прогулки и путешествия""]",avtokreslo-chicco-key1-x-plus-isofix
2,3262,Автокресло Chicco Synthesis XT-Plus,Chicсo,Автокресла,"[""Автокресла"",""Прогулки и путешествия""]",avtokreslo-chicco-synthesis-xt-plus
3,3265,Дом-палатка с туннелем «Океан»,Ching-Ching,Домики и палатки,"[""Домики и палатки"",""Тренажеры и игровые компл...",dom-palatka-s-tunnelem-okean
4,3267,Барабан для малышей,Imaginarium,NaN,[],baraban-dlya-malyshey
...,...,...,...,...,...,...
756,6674,Электронные качели MamaRoo NEW,4moms,Электрокачели,"[""Комната и сон"",""Электрокачели""]",elektronnye-kacheli-mamaroo-new
757,6675,Прогулочная коляска Babyzen Yoyo2 6+,Babyzen,Коляски,[],progulochnaya-kolyaska-babyzen-yoyo2-6
758,6676,Автолюлька Maxi-Cosi Pebble + шасси Yoyo,Maxi-Cosi,NaN,[],avtolyulka-maxi-cosi-pebble-shassi-yoyo
759,6677,"Развивающий коврик ""Солнечный денек""",Tiny Love,NaN,[],razvivayushchiy-kovrik-solnechnyy-denek


In [73]:
new_site_p

,id,name,brand,main_category,categories,slug
0,463480210,Кокон для новорожденных Cocoonababy Red Castle...,Red Castle,Коконы для новорожденных,"[""Коконы для новорожденных""]",matras-red-castle-kokon-dlya-novorozhdennyh-co...
1,463480211,Кокон для новорожденных Matello Cocon Baby Lux...,Matello,Коконы для новорожденных,"[""Коконы для новорожденных""]",kokon-dlya-novorozhdennyh-matello-cocon-baby-l...
2,463480212,Кокон-люлька для новорожденных Farla Baby Shel...,Farla,Коконы для новорожденных,"[""Коконы для новорожденных""]",kokon-lyulka-dlya-novorozhdennyh-farla-baby-sh...
3,463480213,Качели-шезлонг 4moms MamaRoo 4.0 напрокат,4moms,Электрокачели,"[""Качели, шезлонги"",""Электрокачели""]",kacheli-shezlong-4moms-mamaroo-40-naprokat
4,463480214,Качели-шезлонг 4moms MamaRoo 3.0 напрокат,4moms,Электрокачели,"[""Качели, шезлонги"",""Электрокачели""]",kacheli-shezlong-4moms-mamaroo-30-naprokat
...,...,...,...,...,...,...
683,1651502137,Адаптеры для автолюльки Yoyo,BabyZen,Коляски,"[""Коляски""]",adaptery-dlya-avtolyulki-yoyo
684,1655011689,Электрокачели MamaRoo NEW + весы Maman напрокат,Maman##4moms,Акции недели,"[""Акции недели"",""Качели, шезлонги"",""Весы""]",elektrokacheli-mamaroo-new-vesy-maman-naprokat
685,1677458905,Обезьянка на кольцах напрокат,Bright Starts,Музыкальные игрушки,"[""Развивающие игрушки"",""Музыкальные игрушки""]",obezyanka-na-koltsah-naprokat
686,1677460825,"Развивающая игрушка ""Маленькие друзья"" напрокат",Fisher-Price,Музыкальные игрушки,"[""Развивающие игрушки"",""Музыкальные игрушки""]",razvivayuschaya-igrushka-malenkie-druzya-naprokat


In [80]:
old_site_p[old_site_p['id'] == 3314]

,id,name,brand,main_category,categories,slug
33,3314,Электронные качели MamaRoo 3.0,4moms,Электрокачели,"[""Комната и сон"",""Электрокачели""]",elektronnye-kacheli-mamaroo-30


In [76]:
mapping = pd.read_csv('old_site_new_site_products.csv')

In [77]:
mapping

,old_site_id,new_site_id
0,3601,463480211
1,3282,463480213
2,3314,463480214
3,3384,463480224
4,3383,463480225
...,...,...
459,3637,495797436
460,3480,495797660
461,3388,1231113729
462,3658,1432513025


In [81]:
mapping[mapping['old_site_id'] == 3314]

,old_site_id,new_site_id
2,3314,463480214
69,3314,463480461
70,3314,463480462
71,3314,463480463


In [87]:
new_site_p[new_site_p['slug'] == 'elektronnye-kacheli-mamaroo-30']

,id,name,brand,main_category,categories,slug


In [86]:
old_site_p[old_site_p['id'] == 3314]

,id,name,brand,main_category,categories,slug
33,3314,Электронные качели MamaRoo 3.0,4moms,Электрокачели,"[""Комната и сон"",""Электрокачели""]",elektronnye-kacheli-mamaroo-30


NameError: name 'slug_to_pid' is not defined